In [1]:
import numpy as np

### ACTIVATION FUNCTIONS ###        https://ml-cheatsheet.readthedocs.io/en/latest/activation_functions.html

# Sigmoid
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_prime(x): # x already sigmoided  - https://www.codingame.com/playgrounds/59631/neural-network-xor-example-from-scratch-no-libs    
    return x * (1 - x)   #  return sigmoid(x) * (1-sigmoid(x))

# RELU 
def relu(x):
    return np.maximum(0, x)

def relu_prime(x):
    return np.where(x > 0, 1, 0)

# Tanh
def tanh(x):
	return (np.exp(x) - np.exp(-x)) / (np.exp(x) + np.exp(-x))

def tanh_prime(x):
    return 1 - np.power(x, 2)  # x already tanh(x)

In [2]:
np.random.seed(2424)
weights = np.random.uniform(-0.5, 0.5, size=9)

# input: x1, x2  and  weights: w11, w21, wb1, w12, w22, wb2, w3, w4, wb3  -  biases already known = 1

### 1 ###

def xor_net(inputs, weights, activation_function=sigmoid,temp=0):
    inputs = np.array(inputs)
    if temp==0:
        weights[2::3] = 1
    weights = weights.reshape(3,3)
    weights_input_hidden = weights[:2]
    weights_hidden_output = weights[2]
    weights_hidden_output = weights_hidden_output.reshape(3,1)
    
    hidden_layer = activation_function(np.dot(inputs, weights_input_hidden))
    output_layer = activation_function(np.dot(hidden_layer, weights_hidden_output))

    return output_layer

### 2 ###

def error(output_layer):
    target = np.array([[0], [1], [1], [0]])
    return np.mean((output_layer-target)**2)     


### 3 ###

def grdmse(weights, activation_function=sigmoid, activation_function_prime=sigmoid_prime):
    targets = np.array([0, 1, 1, 0])
    inputs = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
    gradient = np.zeros_like(weights)

    for i in range(len(inputs)):
        input, target = inputs[i], targets[i]
      
        weights_input_hidden1 = weights[:6].reshape(2, 3)
        weights_hidden_output = weights[6:].reshape(3, 1)
        
        hidden_layer1 = activation_function(np.dot(input, weights_input_hidden1))
        output = activation_function(np.dot(hidden_layer1, weights_hidden_output))

        output_derror = (output - target) * activation_function_prime(output)
        hidden_layer1_derror = output_derror.dot(weights_hidden_output.T) * activation_function_prime(hidden_layer1)

        gradient[:6] += np.outer(input, hidden_layer1_derror).flatten()
        gradient[6:] += np.outer(hidden_layer1, output_derror).flatten()

    gradient /= len(inputs)   
    return gradient

### Training - 4 ###

def train_xor_net(activation_function=sigmoid, activation_function_prime=sigmoid_prime, learning_rate=0.1, epochs=10000):
    inputs = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
    targets = np.array([[0], [1], [1], [0]])

    weights_input_hidden = np.random.uniform(-0.5, 0.5, size=(2, 3))  
    weights_hidden_output = np.random.uniform(-0.5, 0.5, size=(3, 1))
    weights_input_hidden[0][2] = 1
    weights_input_hidden[1][2] = 1
    weights_hidden_output[2] = 1
    temp=0

    for epoch in range(epochs):
        hidden_layer = activation_function(np.dot(inputs, weights_input_hidden))
        output_layer = activation_function(np.dot(hidden_layer, weights_hidden_output))

        output_derror = (targets - output_layer) * activation_function_prime(output_layer)
        
        hidden_layer_derror = output_derror.dot(weights_hidden_output.T) * activation_function_prime(hidden_layer)

        weights_hidden_output += hidden_layer.T.dot(output_derror) * learning_rate
        weights_input_hidden += inputs.T.dot(hidden_layer_derror) * learning_rate
        
        weights= np.concatenate((weights_input_hidden.flatten(), weights_hidden_output.flatten()))

        loss = error(output_layer)

        misclassified = np.sum((np.round(xor_net(inputs, weights, activation_function,temp)) != targets))
        temp=1

        if epoch % 1000 == 0:
            print(f"For epoch {epoch}: the MSE obtained by our network on the training set is {loss:.4f} and the number of misclassified inputs is {misclassified}")

    trained_weights = np.concatenate((weights_input_hidden.flatten(), weights_hidden_output.flatten()))
    
    print(f"Trained weights with {activation_function.__name__} activation function: {trained_weights}")
    return trained_weights


trained_weights = train_xor_net(learning_rate=0.1)                                                                        # activation function: sigmoid
# trained_weights = train_xor_net(activation_function=tanh, activation_function_prime=tanh_prime, learning_rate=0.1)      # activation function: tanh
# trained_weights = train_xor_net(activation_function=relu, activation_function_prime=relu_prime, learning_rate=0.1)      # activation function: relu


inputs = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
predictions = np.array([xor_net(input, trained_weights,temp=1) for input in inputs])

for i, input in enumerate(inputs):
    print(f"Input: {input}, Prediction: {np.round(predictions[i])}")

For epoch 0: the MSE obtained by our network on the training set is 0.2905 and the number of misclassified inputs is 2
For epoch 1000: the MSE obtained by our network on the training set is 0.2343 and the number of misclassified inputs is 1
For epoch 2000: the MSE obtained by our network on the training set is 0.1864 and the number of misclassified inputs is 1
For epoch 3000: the MSE obtained by our network on the training set is 0.1349 and the number of misclassified inputs is 0
For epoch 4000: the MSE obtained by our network on the training set is 0.0542 and the number of misclassified inputs is 0
For epoch 5000: the MSE obtained by our network on the training set is 0.0250 and the number of misclassified inputs is 0
For epoch 6000: the MSE obtained by our network on the training set is 0.0159 and the number of misclassified inputs is 0
For epoch 7000: the MSE obtained by our network on the training set is 0.0117 and the number of misclassified inputs is 0
For epoch 8000: the MSE obt

In [3]:
np.random.seed(2424)

learning_rates = [0.01, 0.1, 0.2]
activation_functions = [sigmoid, tanh, relu]
activation_primes = [sigmoid_prime, tanh_prime, relu_prime]

for lr in learning_rates:
    for activation_func, activ_prime in zip(activation_functions, activation_primes):
        print(f"Training with learning rate {lr} and activation function {activation_func.__name__} results:")
        trained_weights = train_xor_net(activation_function=activation_func, activation_function_prime=activ_prime, learning_rate=lr)
        inputs = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
        predictions = np.array([xor_net(input, trained_weights, temp=1) for input in inputs])

        for i, input in enumerate(inputs):
            print(f"Input:{input}, Prediction:{np.round(predictions[i])}")
        


Training with learning rate 0.01 and activation function sigmoid results:
For epoch 0: the MSE obtained by our network on the training set is 0.2880 and the number of misclassified inputs is 2
For epoch 1000: the MSE obtained by our network on the training set is 0.2476 and the number of misclassified inputs is 1
For epoch 2000: the MSE obtained by our network on the training set is 0.2467 and the number of misclassified inputs is 1
For epoch 3000: the MSE obtained by our network on the training set is 0.2458 and the number of misclassified inputs is 1
For epoch 4000: the MSE obtained by our network on the training set is 0.2447 and the number of misclassified inputs is 1
For epoch 5000: the MSE obtained by our network on the training set is 0.2432 and the number of misclassified inputs is 1
For epoch 6000: the MSE obtained by our network on the training set is 0.2414 and the number of misclassified inputs is 1
For epoch 7000: the MSE obtained by our network on the training set is 0.23